# Creazione dei grafi

## Autore - autore

### Ulteriore preprocessing

In [16]:
import networkx as nx
import pandas as pd
import json
import matplotlib.pyplot as plt

In [17]:
with open('Data/merged_authors.json', 'r') as f:
        merged_authors = json.load(f)

In [18]:
merged_authors

{'Alessandro Liberati': {'collaborators_by_paper': {'Going from evidence to recommendations': [[],
    'Computer science'],
   'GRADE guidelines: 10. Considering resource use and rating the quality of economic evidence': [['Silvia Pregno',
     'Massimo Brunetti'],
    'Computer science'],
   'Reprint—Preferred Reporting Items for Systematic Reviews and Meta-Analyses: The PRISMA Statement': [[],
    'Medicine'],
   'Incorporating considerations of resources use into grading recommendations': [['Holger J. Schünemann'],
    'Computer science'],
   'What contributions do languages other than English make on the results of meta-analyses?': [[],
    'Medicine'],
   'Preferred reporting items for systematic review and meta-analysis protocols (PRISMA-P) 2015: elaboration and explanation': [[],
    'Computer science'],
   'Preferred reporting items for systematic reviews and meta-analyses: the PRISMA statement': [[],
    'Computer science']},
  'anno': {'2000': ['Mario Negri Institute for Phar

In [19]:
def create_df(dictionary):
    set_collaboratori = set()
    tot_collaborazioni = []

    for autore, dati_autore in dictionary.items():
        collaboratori_per_paper = dati_autore.get('collaborators_by_paper', {})

        for paper, infos in collaboratori_per_paper.items():

            collaboratori = infos[0]

            for i in collaboratori: 
                if (autore, i) in set_collaboratori:
                    continue

                else:
                    tot_collaborazioni.append((autore, i))
                    set_collaboratori.add((autore, i))

    return tot_collaborazioni

In [20]:
tot_collaborazioni = create_df(merged_authors)

In [21]:
df_collaborazioni = pd.DataFrame(tot_collaborazioni, columns= ['Autore', 'Collaboratore'])

In [22]:
nodi_source = set(df_collaborazioni['Autore'].unique())
nodi_target = set(df_collaborazioni['Collaboratore'].unique())
tutti_nodi = nodi_source.union(nodi_target)
print(f"Nodi univoci totali: {len(tutti_nodi)}")

print(f"NaN in Autore: {df_collaborazioni['Autore'].isna().sum()}")
print(f"NaN in Collaboratore: {df_collaborazioni['Collaboratore'].isna().sum()}")

print(df_collaborazioni['Autore'].dtype, df_collaborazioni['Collaboratore'].dtype)

Nodi univoci totali: 102965
NaN in Autore: 0
NaN in Collaboratore: 0
object object


Probabilmente il grafo presenta un numero maggiore di nodi rispetto alla lunghezza del dizionario perché sono presenti collaboratori che non appaiono nella sezione autori. 

In [23]:
filename = 'Data/grafo.csv'

import networkx as nx

# Aggiungi gli archi dal DataFrame
graph = nx.from_pandas_edgelist(df_collaborazioni, source = 'Autore', target = "Collaboratore")

In [24]:
print(graph)

Graph with 102965 nodes and 635613 edges


In [25]:
nx.write_edgelist(graph, filename, data= False, delimiter=",")

In [26]:
g = nx.read_edgelist(filename, delimiter=",")

In [27]:
print(g)

Graph with 102965 nodes and 635613 edges


## Autore - istituzione

In [28]:
# Costruzione del grafo bipartito autore/istituzioni con anno come attributo del link
grafo_autori_istituzioni = nx.Graph()

for author, info in merged_authors.items():
    anni_affiliazioni = info.get("anno", {})

    for anno, istituzioni in anni_affiliazioni.items():
        for istituzione in istituzioni:
            grafo_autori_istituzioni.add_node(author, tipo="autore")
            grafo_autori_istituzioni.add_node(istituzione, tipo="istituzione")
            grafo_autori_istituzioni.add_edge(author, istituzione, anno=int(anno))

In [29]:
for u, v, d in grafo_autori_istituzioni.edges(data=True):
    print(u, "<-->", v, "anno:", d["anno"])

Alessandro Liberati <--> Mario Negri Institute for Pharmacological Research anno: 2009
Alessandro Liberati <--> Agenzia Sanitaria e Sociale Regionale anno: 2012
Alessandro Liberati <--> University of Modena and Reggio Emilia anno: 2015
Mario Negri Institute for Pharmacological Research <--> Ettore Beghi anno: 2022
Mario Negri Institute for Pharmacological Research <--> Annamaria Vezzani anno: 2017
Mario Negri Institute for Pharmacological Research <--> Riccardo Brambilla anno: 2016
Mario Negri Institute for Pharmacological Research <--> Silvia Balosso anno: 2013
Mario Negri Institute for Pharmacological Research <--> Teresa Ravizza anno: 2017
Mario Negri Institute for Pharmacological Research <--> Silvia Minozzi anno: 2020
Mario Negri Institute for Pharmacological Research <--> Elisa R. Zanier anno: 2019
Mario Negri Institute for Pharmacological Research <--> Gianluigi Forloni anno: 2019
Mario Negri Institute for Pharmacological Research <--> Laura Reale anno: 2017
Mario Negri Institut

In [30]:
#nx.write_gexf(G, "autori_istituzioni.gexf")

In [31]:
name_autori_istituzioni = "Data/grafo_autori_istituzioni.csv"
nx.write_edgelist(grafo_autori_istituzioni, name_autori_istituzioni, data= False, delimiter=",")

### PROVIAMO ALTRO METODO

In [32]:
df_collaborazioni

,Autore,Collaboratore
0,Alessandro Liberati,Silvia Pregno
1,Alessandro Liberati,Massimo Brunetti
2,Alessandro Liberati,Holger J. Schünemann
3,Nicola De Stefano,Marco Battaglini
4,Nicola De Stefano,Maria De Luca
...,...,...
848572,Paolo Caravaggi,Alberto Leardini
848573,Vittoria Nardone,Fabio Martinelli
848574,Vittoria Nardone,Francesco Mercaldo
848575,Vittoria Nardone,Antonella Santone


In [33]:
dict_autori = df_collaborazioni.groupby('Autore')['Collaboratore'].apply(list).to_dict()

In [34]:
dict_autori

{'A Agostoni': ['Massimo Cugno',
  'Marco Cicardi',
  'Luigi Bergamaschini',
  'Lorenza C. Zingale'],
 'A Agresti': ['Francesco De Marchis',
  'Alessandro Preti',
  'Marco E. Bianchi',
  'Emilie Vénéreau',
  'Andrea Gorzanelli',
  'Antonella Antonelli',
  'Roberta Vertemara',
  'Annamaria Vezzani',
  'Valentina Iori',
  'Teresa Ravizza',
  'Mirjana Carli',
  'Massimo Rizzi',
  'Mattia Maroso'],
 'A Alberti': ['Emanuela Avola',
  'Mirella Vinci',
  'Corrado Romano',
  'Stefania Giusto',
  'Serafino Buono',
  'Marco Fichera',
  'Pinella Failla',
  'Paolo Bosco',
  'Rosa Pettinato',
  'Lucia Grillo',
  'Carmela Scuderi'],
 'A Bertelli': ['Maddalena Corsini',
  'Giulia Baini',
  'Elisabetta Miraldi',
  'Giorgio Cappellucci',
  'Marco Biagi'],
 'A Bertini': ['Cristina Consani',
  'Daniele Taccola',
  'P Rindi',
  'Vincenzo Panichi',
  'Umberto Maggiore',
  'M. Migliori',
  'Giovanni Manca Rizza',
  'Davide Giorgi',
  'Achille C. Pessina',
  'C. Ganzaroli',
  'R Scognamiglio',
  'Mario Maria

In [35]:
graph_tentativo2 = nx.Graph()

In [36]:
for k, v in dict_autori.items(): 
    graph_tentativo2.add_node(k)

    for collaborator in set(v):
        graph_tentativo2.add_node(collaborator)
        graph_tentativo2.add_edge(k, collaborator)

In [37]:
print(graph_tentativo2)

Graph with 102965 nodes and 635613 edges


In [38]:
num_connected_components = nx.number_connected_components(graph_tentativo2)
ccomponents = sorted(nx.connected_components(graph_tentativo2), key = len, reverse = True)

gcc = ccomponents[0]
print(F'Il great connected component contiene il {(len(gcc)/g.number_of_nodes())*100}% dei nodi')

Il great connected component contiene il 96.54639926188511% dei nodi
